# AI-Based Medical Image Segmentation using U-Net
This notebook demonstrates a beginner-friendly implementation.

In [ ]:
# Import required libraries
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models


## 1. Preprocessing Module

In [ ]:
# Load and preprocess image
def preprocess_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (256,256))
    img = img / 255.0  # normalization
    img = cv2.GaussianBlur(img, (5,5), 0)  # noise removal
    img = np.expand_dims(img, axis=-1)
    return img

# Example usage (replace with your image path)
# image = preprocess_image('mri_image.png')


## 2-4. U-Net Model (Encoder + Bottleneck + Decoder)

In [ ]:
# Convolution block
def conv_block(x, filters):
    x = layers.Conv2D(filters, (3,3), padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, (3,3), padding='same', activation='relu')(x)
    return x

# Build U-Net
def build_unet():
    inputs = layers.Input((256,256,1))

    # Encoder
    c1 = conv_block(inputs, 64)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 128)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 256)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 512)
    p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    bn = conv_block(p4, 1024)

    # Decoder
    u1 = layers.UpSampling2D()(bn)
    u1 = layers.concatenate([u1, c4])
    c5 = conv_block(u1, 512)

    u2 = layers.UpSampling2D()(c5)
    u2 = layers.concatenate([u2, c3])
    c6 = conv_block(u2, 256)

    u3 = layers.UpSampling2D()(c6)
    u3 = layers.concatenate([u3, c2])
    c7 = conv_block(u3, 128)

    u4 = layers.UpSampling2D()(c7)
    u4 = layers.concatenate([u4, c1])
    c8 = conv_block(u4, 64)

    # Output layer
    outputs = layers.Conv2D(1, (1,1), activation='sigmoid')(c8)

    model = models.Model(inputs, outputs)
    return model

model = build_unet()
model.summary()


## Compile Model

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## Dummy Training (Example)

In [ ]:
# Dummy data for demo
X = np.random.rand(10,256,256,1)
y = np.random.randint(0,2,(10,256,256,1))

model.fit(X, y, epochs=1)


## 5. Boundary Optimization

In [ ]:
def post_process(mask):
    mask = (mask > 0.5).astype(np.uint8)
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

# Example
# pred = model.predict(X)[0]
# final_mask = post_process(pred)
